# GAS-BayesSHAP — high-dimensional sub-enumerative certification (M=30)

The latest audit's decisive question: the M=11 nominal-certification runs were **post-enumerative** (2048 unique coalition evals = 2^11 — the full power set was cached before the certificate closed).  This notebook runs the same finite-population machinery at **M=30** where 2^30 ≈ 1.07e9 is infeasible, on a sparse synthetic game with **closed-form exact Shapley values** (so RMSE and sign validation stay checkable), and reports unique coalition evals vs 2^M, sign-certified features, and the nominal-certification status.  Orchestrates `scripts/probe_high_dim.py` only — no duplicated algorithm.

## 0. Environment & config

In [1]:
import sys, os, time, json, subprocess
from pathlib import Path
sys.path.insert(0, "..")
import gas_bayesshap

ROOT = Path("..").resolve()
SCRIPTS = ROOT / "scripts"
print("GAS-BayesSHAP", gas_bayesshap.__version__)

M       = int(os.environ.get("HM_M", "30"))
GRID    = os.environ.get("HM_GRID", "20000,50000,100000")
HIGHK   = os.environ.get("HM_HIGHK", "500000,1000000")
SKIP    = set(os.environ.get("GAS_SKIP", "").split(",")) - {""}

def run(*args, tag="", skip=False):
    if skip:
        print(f"--- SKIPPED: {tag or ' '.join(args)}"); return 0.0
    cmd = [sys.executable, str(SCRIPTS / args[0]), *args[1:]]
    t0 = time.time()
    print(f"\n>>> {tag or ' '.join(args)}")
    r = subprocess.run(cmd, cwd=ROOT)
    dt = time.time() - t0
    if r.returncode != 0:
        raise RuntimeError(f"FAILED ({r.returncode}): {' '.join(args)}")
    print(f"<<< done in {dt/60:.1f} min")
    return dt

print(f"M={M} GRID={GRID} HIGHK={HIGHK} SKIP={sorted(SKIP)}")

GAS-BayesSHAP 11.0.0
M=30 GRID=20000,50000,100000 HIGHK=500000,1000000 SKIP=[]


## A. M=30 grid — finite-population, K ∈ {20k, 50k, 100k}

Fast grid (~3 min).  Expected: unique coalition evals ~2e4–9e4 ≪ 2^30 (ratio ~1e-4), point RMSE ~1e-5, widths shrinking as ~1/√K; `certificate_at_nominal_level=False` (coupon budget over C(29,s) pairs cannot close at these K).

In [2]:
run("probe_high_dim.py", "--M", str(M), "--budgets", GRID,
    "--mode", "finite_population", tag=f"A. M={M} fp grid K={GRID}",
    skip="A" in SKIP)


>>> A. M=30 fp grid K=20000,50000,100000
  M=30 K=  20000: BUDGET_EXHAUSTED conv=False at_nom=False level=0.0 unique=20802 unique/2^M=1.94e-05 sign_cert=0 signs_ok=1 rmse=7.92e-05 W=0.4132 (8s)
  M=30 K=  50000: BUDGET_EXHAUSTED conv=False at_nom=False level=0.0 unique=47644 unique/2^M=4.44e-05 sign_cert=0 signs_ok=1 rmse=4.01e-05 W=0.1735 (17s)
  M=30 K= 100000: BUDGET_EXHAUSTED conv=False at_nom=False level=0.0 unique=91274 unique/2^M=8.50e-05 sign_cert=0 signs_ok=1 rmse=1.32e-05 W=0.0920 (33s)

done in 58s; main_results/paper_high_dim_M30_finite_population_summary.csv + paper_high_dim_M30_summary.csv
<<< done in 1.0 min


58.318557024002075

## B. M=30 sign-cert search — K ∈ {500k, 1e6}

The widths at K=1e5 (~0.09) are above the driver attributions (|φ|≈0.033–0.037); the width law predicts sign certification of the driver features somewhere in K ≈ 5e5–1e6.  **~15–25 min.**  If it appears, it is an *empirical-event* sign certification (the certified interval is not a nominal 1−δ certificate — the coupon stays open); the notebook reports that distinction explicitly.

In [3]:
run("probe_high_dim.py", "--M", str(M), "--budgets", HIGHK,
    "--mode", "finite_population", tag=f"B. M={M} fp high-K {HIGHK}",
    skip="B" in SKIP)


>>> B. M=30 fp high-K 500000,1000000
  M=30 K= 500000: VALID            conv=True at_nom=False level=0.0 unique=229850 unique/2^M=2.14e-04 sign_cert=3 signs_ok=1 rmse=9.16e-06 W=0.0381 (88s)
  M=30 K=1000000: VALID            conv=True at_nom=False level=0.0 unique=229850 unique/2^M=2.14e-04 sign_cert=3 signs_ok=1 rmse=9.16e-06 W=0.0381 (86s)

done in 175s; main_results/paper_high_dim_M30_finite_population_summary.csv + paper_high_dim_M30_summary.csv
<<< done in 2.9 min


175.19022393226624

## C. Spec-range contrast at K=100k

One spec-range row for the width ratio (spec ≈ 20× fp at the same budget), confirming the empirical range is what makes high-dim sign certification feasible at all.  ~2 min.

In [4]:
run("probe_high_dim.py", "--M", str(M), "--budgets", "100000",
    "--mode", "spec", tag=f"C. M={M} spec contrast K=100k",
    skip="C" in SKIP)


>>> C. M=30 spec contrast K=100k
  M=30 K= 100000: BUDGET_EXHAUSTED conv=False at_nom=False level=None unique=91274 unique/2^M=8.50e-05 sign_cert=0 signs_ok=1 rmse=1.32e-05 W=3.1118 (33s)

done in 33s; main_results/paper_high_dim_M30_spec_summary.csv + paper_high_dim_M30_summary.csv
<<< done in 0.6 min


33.59189295768738

## D. Summary — sub-enumerative? sign-cert? nominal?

In [5]:
import pandas as pd
import numpy as np
p = ROOT / "main_results" / f"paper_high_dim_M{M}_summary.csv"
if not p.exists():
    print(f"NOT FOUND: {p.name} — run sections A–C first")
else:
    d = pd.read_csv(p)
    cols = ["K", "range_mode", "status", "converged",
            "certificate_at_nominal_level", "realised_coverage_level",
            "unique_coalition_evals", "unique_vs_2M_ratio",
            "n_sign_certified", "signs_match_exact", "rmse_vs_exact",
            "mean_width"]
    print(d[cols].to_string(index=False))
    fp = d[d.range_mode == "finite_population"]
    print("\nHONEST REPORT:")
    print(f"  unique/2^M ratio: {fp['unique_vs_2M_ratio'].min():.2e} .. "
          f"{fp['unique_vs_2M_ratio'].max():.2e}  (sub-enumerative if << 1)")
    print(f"  max unique: {int(fp['unique_coalition_evals'].max())} vs 2^M = {2**M}")
    print(f"  sign-certified runs: {int((fp['n_sign_certified'] > 0).sum())} "
          f"| signs validated: {bool(fp['signs_match_exact'].all())}")
    print(f"  at_nominal_level anywhere: {bool(fp['certificate_at_nominal_level'].any())} "
          "(coupon wall at M=30 — expected False; the rigorous nominal "
          "certificate is near-enumerative, stated honestly in the paper)")

      K        range_mode           status  converged  certificate_at_nominal_level  realised_coverage_level  unique_coalition_evals  unique_vs_2M_ratio  n_sign_certified  signs_match_exact  rmse_vs_exact  mean_width
 500000 finite_population            VALID       True                         False                      0.0                  229850            0.000214                 3                  1       0.000009    0.038131
1000000 finite_population            VALID       True                         False                      0.0                  229850            0.000214                 3                  1       0.000009    0.038131
 100000              spec BUDGET_EXHAUSTED      False                         False                      NaN                   91274            0.000085                 0                  1       0.000013    3.111783

HONEST REPORT:
  unique/2^M ratio: 2.14e-04 .. 2.14e-04  (sub-enumerative if << 1)
  max unique: 229850 vs 2^M = 1073741824
  sign-

## Expected runtime and honest notes
- **Full run ≈ 25–35 min** (A ~3 min, B ~15–25 min, C ~2 min).
- **Smoke:** `GAS_SKIP=B,C HM_GRID=5000` (~30 s).
- The M=11 nominal-certification runs remain **post-enumerative** (2048 unique = 2^11) — this notebook does NOT change that; it establishes what happens when 2^M is infeasible.
- The M=30 result will likely show: sub-enumerative point-estimate fidelity (unique ≪ 2^30), empirical-range sign certification of drivers at high K, but `certificate_at_nominal_level=False` — the coupon-collector budget over C(M-1,s) pairs is the honest scaling wall.  Report it exactly as it lands.
- Commit the resulting `paper_high_dim_M{M}_summary.csv`.